# Retail Sales Forecasting & Business Intelligence — Milestone 3
**Participant:** Participant_M3_01  
**Project:** Retail Sales Forecasting & Business Intelligence  
**Milestone:** 3  
**Dataset package:** `Retail_(TS).zip`

## Objective
Forecast daily sales for each store-item combination and support inventory/replenishment decisions.

## Milestone scope
prepare chronological sequences for LSTM/GRU or feed-forward network; scale on training data only; compare with ML and simple baselines on same holdout period.

> This is a synthetic sample submission for evaluator testing. Dataset files must be placed in the paths referenced below. Results are not pre-filled or fabricated.

## LSTM-style forecasting experiment
Create sequences in chronological order and fit scaling only on the training portion. A simple baseline and ML comparison are required before claiming deep learning improves forecasts.

In [ ]:
from pathlib import Path
import pandas as pd, numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor

df = pd.read_csv(Path("data")/"retail_data.csv", parse_dates=["date"])
series = (df.groupby("date", as_index=False)["sales"].sum().sort_values("date"))
values = series["sales"].to_numpy(dtype=float)
split = int(len(values)*0.8)
scaler = MinMaxScaler()
train_scaled = scaler.fit_transform(values[:split].reshape(-1,1)).ravel()
test_scaled = scaler.transform(values[split:].reshape(-1,1)).ravel()

def make_sequences(arr, lookback=14):
    X, y = [], []
    for i in range(lookback, len(arr)):
        X.append(arr[i-lookback:i]); y.append(arr[i])
    return np.asarray(X), np.asarray(y)
lookback=14
Xtr,ytr=make_sequences(train_scaled,lookback)
combined=np.concatenate([train_scaled[-lookback:],test_scaled])
Xte,yte=make_sequences(combined,lookback)
# MLP is a lightweight neural baseline; replace with LSTM/GRU for the full brief.
nn=MLPRegressor(hidden_layer_sizes=(64,32),early_stopping=True,max_iter=200,random_state=42).fit(Xtr,ytr)
pred_scaled=nn.predict(Xte)
pred=scaler.inverse_transform(pred_scaled.reshape(-1,1)).ravel()
actual=scaler.inverse_transform(yte.reshape(-1,1)).ravel()
print("MLP MAE",mean_absolute_error(actual,pred),"RMSE",mean_squared_error(actual,pred)**.5)

## Comparison and limitations
This aggregates sales across all stores/items for a compact demonstration; the full project should forecast each store-item series and compare against ML and naive/seasonal-naive baselines on the same holdout period.